## Step 8b: Evaluation Protocol (one scoring rule, one AUC definition, labelled operating points)

Earlier versions mixed several final-score rules, AUC negative sets and thresholds. Everything below is defined once here and used by every table and figure.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# EVALUATION PROTOCOL — ONE definition shared by every table and figure
# (reviewer §3A, §3C, §3E, §3F).  Earlier versions mixed several scoring rules
# (last-round, last-3-round mean, mean over all rounds incl. neutral fills), several AUC
# negative sets (all-other clients vs honest only) and per-seed vs seed-averaged scores.
# ═════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import roc_auc_score as _roc_auc_score

# (1) FINAL CLIENT SCORE — one rule for every method, every table, every replay:
#     mean of the client's last FINAL_K scores taken in rounds in which it PARTICIPATED.
#     A client never observed gets the median of the observed clients ("no evidence").
def final_signal(round_scores, k=None):
    k = FINAL_K if k is None else k
    R, N = round_scores.shape
    out = np.full(N, np.nan)
    for c in range(N):
        v = round_scores[:, c]; v = v[~np.isnan(v)]
        if len(v): out[c] = v[-k:].mean()
    if np.isnan(out).any():
        out[np.isnan(out)] = np.nanmedian(out) if (~np.isnan(out)).any() else 0.5
    return out

# (2) AUC — positives = the attacker set under test; negatives = the HONEST clients only.
#     (The old per-attack AUC used all 29 other clients, i.e. the other attackers too.)
def auc_vs_honest(scores, attacker_ids, honest_ids=None):
    honest_ids = HONEST_CLIENTS if honest_ids is None else honest_ids
    ids = list(attacker_ids) + list(honest_ids)
    y   = [1]*len(attacker_ids) + [0]*len(honest_ids)
    s   = np.asarray(scores, dtype=float)[ids]
    if np.isnan(s).any(): return float('nan')
    return float(_roc_auc_score(y, s))

def auc_pooled(scores):      return auc_vs_honest(scores, MALICIOUS)
def auc_per_type(scores):
    return dict(overt=auc_vs_honest(scores, POISONED_CLIENTS),
                slow=auc_vs_honest(scores, SLOW_CLIENTS),
                inversion=auc_vs_honest(scores, INVERSE_CLIENTS))

def pooled_identity_check(scores, tol=1e-9):
    """With the SAME honest negatives for every attack type, the pooled attacker-vs-honest
    AUC equals the count-weighted mean of the per-type AUCs (1 overt, 3 slow, 1 inversion).
    Returns (pooled, weighted_mean, identity_holds)."""
    per = auc_per_type(scores)
    w   = dict(overt=len(POISONED_CLIENTS), slow=len(SLOW_CLIENTS), inversion=len(INVERSE_CLIENTS))
    weighted = sum(w[k]*per[k] for k in per) / sum(w.values())
    pooled   = auc_pooled(scores)
    return pooled, weighted, abs(pooled - weighted) < tol

def round_auc(scores, mask, attacker_ids):
    """Within-round AUC restricted to clients that participated in that round (NaN if the
    round contains no attacker of this type or no honest client)."""
    pos = [a for a in attacker_ids if mask[a]]
    neg = [h for h in HONEST_CLIENTS if mask[h]]
    if not pos or not neg: return float('nan')
    y = [1]*len(pos) + [0]*len(neg)
    return float(_roc_auc_score(y, [scores[i] for i in pos + neg]))

# (3) OPERATING POINTS — labelled separately everywhere they appear
def flag_fixed(scores, thr=None):
    thr = FLAG_THRESHOLD if thr is None else thr
    return (np.asarray(scores) >= thr).astype(int)
def flag_topq(scores, pct=70):
    return (np.asarray(scores) >= np.percentile(scores, pct)).astype(int)

OPERATING_POINTS = {
 'OP-A': dict(name='OP-A  fixed threshold tau = FLAG_THRESHOLD (PRINCIPAL)',
              note='pre-declared, does not use the contamination rate; also the rule written to the ledger',
              fn=lambda s: flag_fixed(s)),
 'OP-B': dict(name='OP-B  top-30% (70th-percentile) rule',
              note='data-dependent: always flags 9 of 30 clients, i.e. implicitly assumes a contamination rate',
              fn=lambda s: flag_topq(s, 70)),
}
SENSITIVITY_THRESHOLDS = ABLATION_THRESHOLDS      # OP-S: 0.50 / 0.60 / 0.70, sensitivity analysis only

# (4) CONFUSION COUNTS — precision/recall/F1 alone cannot show "nobody was flagged"
def confusion_counts(gt, pred):
    gt, pred = np.asarray(gt).astype(int), np.asarray(pred).astype(int)
    TP = int(((pred==1)&(gt==1)).sum()); FP = int(((pred==1)&(gt==0)).sum())
    TN = int(((pred==0)&(gt==0)).sum()); FN = int(((pred==0)&(gt==1)).sum())
    nf = TP + FP
    return dict(TP=TP, FP=FP, TN=TN, FN=FN, n_flagged=nf,
                precision=(TP/nf if nf else float('nan')),           # undefined if nothing flagged
                recall=(TP/(TP+FN) if (TP+FN) else float('nan')),
                f1=(2*TP/(2*TP+FP+FN) if (2*TP+FP+FN) else float('nan')))

def fmt_counts(c):
    p = 'undef (0 flagged)' if np.isnan(c['precision']) else f"{c['precision']:.3f}"
    return (f"TP={c['TP']} FP={c['FP']} TN={c['TN']} FN={c['FN']} | flagged={c['n_flagged']} | "
            f"P={p} R={c['recall']:.3f} F1={c['f1']:.3f}")

# (5) Per-method AUC table (mean ± std over seeds; per-seed AUC, NOT AUC of seed-averaged scores)
AUC_METHODS = METHOD_ORDER + ['TemporalConsistency']
def method_auc_table(results, methods=None):
    methods = AUC_METHODS if methods is None else methods
    out = {}
    for m in methods:
        rows = []
        for r in results:
            sig = r[6][m]
            pt  = auc_per_type(sig)
            rows.append([pt['overt'], pt['slow'], pt['inversion'], auc_pooled(sig)])
        a = np.array(rows)
        out[m] = dict(mean=a.mean(0), std=a.std(0), per_seed=a)
    return out            # columns: overt, slow, inversion, pooled

# (6) Attack-impact metrics (overall accuracy hides targeted damage)
def attack_metrics(params, X_test, y_test, input_dim, n_classes=N_CLASSES):
    _, probs = evaluate_model(params, X_test, y_test, input_dim, n_classes)
    pred = probs.argmax(1); out = {'acc': float((pred == y_test).mean())}
    hit = tot = 0
    for src, dst, nm in [(POISON_FROM, POISON_TO, '0to1'), (POISON_FROM_2, POISON_TO_2, '2to3')]:
        m = (y_test == src)
        out[f'src_acc_{nm}'] = float((pred[m] == src).mean()) if m.any() else float('nan')
        out[f'asr_{nm}']     = float((pred[m] == dst).mean()) if m.any() else float('nan')
        hit += int((pred[m] == dst).sum()); tot += int(m.sum())
    out['targeted_mis_rate'] = hit / max(tot, 1)      # pooled over both flipped source classes
    return out

# (7) Protocol registry — what every table in the manuscript is computed with
PROTOCOL_REGISTRY = [
 ('Table IV (per-attack AUC)', 'main FL run, all 6 methods end-to-end, seeds=SEEDS',
  'final_signal (last FINAL_K observed scores)', 'AUC vs 25 honest clients, per seed → mean±std', 'none (AUC)'),
 ('Table V (accuracy)', 'same main FL run', 'n/a', 'test accuracy after the last executed round', 'n/a'),
 ('Table VI (component ablation)', 'REPLAY of the recorded main-run trust/inversion/participation signals',
  'same final_signal', 'same AUC; Full-CRTAT row is asserted equal to Table IV', 'OP-A and OP-B reported separately'),
 ('Table VII (trust/ledger ablation)', 'RE-RUN of the CRTAT branch only, same seeds/splits/RNG streams',
  'same final_signal', 'same AUC', 'OP-A and OP-B reported separately'),
 ('Fig. 15 / pooled AUC', 'replay (as Table VI)', 'same final_signal',
  '5 attackers vs 25 honest; identity with per-type weighted mean is checked', 'none'),
]
def print_protocol_registry():
    print('EVALUATION PROTOCOL REGISTRY')
    print('═'*118)
    for row in PROTOCOL_REGISTRY:
        print(f'  {row[0]:<32} experiment : {row[1]}')
        print(f'  {"":<32} score      : {row[2]}')
        print(f'  {"":<32} AUC        : {row[3]}')
        print(f'  {"":<32} threshold  : {row[4]}')
        print('  ' + '─'*114)
    print(f'  Operating points: {OPERATING_POINTS["OP-A"]["name"]}  tau={FLAG_THRESHOLD}')
    print(f'                    {OPERATING_POINTS["OP-B"]["name"]}')
    print(f'                    OP-S sensitivity thresholds {SENSITIVITY_THRESHOLDS} (sensitivity analysis only)')
    print(f'  FINAL_K={FINAL_K} | neutral trust for absent rounds (original mode) = {NEUTRAL_TRUST} | '
          f'early stopping = {USE_EARLY_STOPPING}')
print_protocol_registry()


EVALUATION PROTOCOL REGISTRY
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Table IV (per-attack AUC)        experiment : main FL run, all 6 methods end-to-end, seeds=SEEDS
                                   score      : final_signal (last FINAL_K observed scores)
                                   AUC        : AUC vs 25 honest clients, per seed → mean±std
                                   threshold  : none (AUC)
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Table V (accuracy)               experiment : same main FL run
                                   score      : n/a
                                   AUC        : test accuracy after the last executed round
                                   threshold  : n/a
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Table VI (c